In [1]:
import arcpy
from datetime import datetime
from collections import defaultdict
import csv
from collections import Counter

import pandas as pd
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
from arcgis.features import GeoAccessor

In [2]:
#PORTAL BAĞLANTISI
# --- Portal bağlantı bilgileri ---
portal_url = "https://geo.ipa.istanbul/portal"
username = "harun.karadoru"
password = "harun.karadoru1"

# --- Feature Service (0. layer!) ---
feature_service_url = (
    "https://geo.ipa.istanbul/server/rest/services/"
    "Hosted/detayli_ak/FeatureServer/0"
)

# --- Lokal GDB ---
gdb_path = r"C:\Users\harun.korucu\Desktop\veri_temin_harun\detayli_ak_local.gdb"

# --- Tarih (YYYYMMDD) ---
today = datetime.now().strftime("%Y%m%d")

# --- Çıktı feature class adı ---
out_fc_name = f"detayli_ak_{today}"

# --- Ortam ayarları ---
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True

# --- Portal'a bağlan ---
arcpy.SignInToPortal(portal_url, username, password)
print("Portal bağlantısı OK:", arcpy.GetActivePortalURL())

# --- Portal verisini lokale yaz ---
arcpy.conversion.FeatureClassToFeatureClass(
    in_features=feature_service_url,
    out_path=gdb_path,
    out_name=out_fc_name
)

print(f"Referans veri oluşturuldu: {out_fc_name}")
print("Kayıt sayısı:", arcpy.management.GetCount(out_fc_name)[0])


Portal bağlantısı OK: https://geo.ipa.istanbul/portal/
Referans veri oluşturuldu: detayli_ak_20260223
Kayıt sayısı: 184284


In [3]:
# REFERANS İLÇE FC
# --- Girdi ---
fc = out_fc_name  # önceki hücreden gelen lokal FC

# --- Tarih ---
today = datetime.now().strftime("%Y%m%d")

# --- İlçe bazlı alt_kullanim istatistiği ---
ilce_stats = defaultdict(lambda: {"total": 0, "null_alt": 0})

with arcpy.da.SearchCursor(fc, ["ilce", "alt_kullanim"]) as cursor:
    for ilce, alt_kullanim in cursor:
        ilce_stats[ilce]["total"] += 1
        if alt_kullanim in (None, "", " "):
            ilce_stats[ilce]["null_alt"] += 1

# --- Referans ilçeler ---
referans_ilceler = [
    ilce for ilce, stats in ilce_stats.items()
    if stats["null_alt"] == 0
]

print("Referans ilçe sayısı:", len(referans_ilceler))

# --- SQL WHERE oluştur ---
# ilce alanı sayısal varsayımıyla
ilce_list_sql = ",".join(str(i) for i in referans_ilceler)
where_clause = f"ilce IN ({ilce_list_sql})"

# --- Çıktı FC adı ---
out_ref_fc = f"detayli_ak_referans_ilceler_{today}"

# --- Referans ilçeleri FC olarak yaz ---
arcpy.conversion.FeatureClassToFeatureClass(
    in_features=fc,
    out_path=arcpy.env.workspace,
    out_name=out_ref_fc,
    where_clause=where_clause
)

print("Referans ilçe feature class oluşturuldu:", out_ref_fc)
print("Kayıt sayısı:", arcpy.management.GetCount(out_ref_fc)[0])


Referans ilçe sayısı: 6
Referans ilçe feature class oluşturuldu: detayli_ak_referans_ilceler_20260223
Kayıt sayısı: 31796


In [4]:
# REFERANS İLÇE FONKSİYON KIRILIMI
# --- Referans FC ---
fc_ref = out_ref_fc

# --- Domain CSV yolu ---
domain_csv = r"C:\Users\harun.korucu\Desktop\veri_temin_harun\scripts\domains.csv"

# --- CSV → sözlük (kod → açıklama) ---
alt_domain = {}

with open(domain_csv, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        alt_domain[row["Kod"]] = row["Adi"]

# --- Toplam kayıt ---
total = int(arcpy.management.GetCount(fc_ref)[0])

values = []

with arcpy.da.SearchCursor(fc_ref, ["alt_kullanim"]) as cursor:
    for (alt_kullanim,) in cursor:
        if alt_kullanim not in (None, "", " "):
            values.append(str(alt_kullanim))  # CSV string olduğu için

freq = Counter(values)

print("REFERANS KÜME – ALT_KULLANIM PROFİLİ (AÇIKLAMALI)")
print("Toplam kayıt:", total)
print("\nKod | Açıklama | Adet | %\n")

for k, v in freq.most_common():
    aciklama = alt_domain.get(k, "AÇIKLAMA YOK")
    print(f"{k} | {aciklama} | {v} | %{(v / total) * 100:.2f}")


REFERANS KÜME – ALT_KULLANIM PROFİLİ (AÇIKLAMALI)
Toplam kayıt: 31796

Kod | Açıklama | Adet | %

1 | KENTSEL KONUT ALANI | 10936 | %34.39
8 | TİCARET-KONUT ALANI | 7772 | %24.44
35 | REFÜJ/ŞEV-YAMAÇ/ÇAYIR | 3194 | %10.05
11 | SANAYİ ALANI | 1433 | %4.51
3 | TİCARET ALANI | 1295 | %4.07
120 | YAPILAŞMAMIŞ ALAN | 1059 | %3.33
30 | PARK | 659 | %2.07
20 | SANAYİ-TİCARET ALANI | 635 | %2.00
48 | CAMİ/MESCİT ALANI | 451 | %1.42
119 | İNŞAAT ALANI | 425 | %1.34
21 | SANAYİ-TİCARET-KONUT ALANI | 263 | %0.83
80 | AÇIK OTOPARK | 261 | %0.82
19 | SANAYİ-KONUT ALANI | 255 | %0.80
26 | KAMU KURUMU (YEREL İDARE) | 188 | %0.59
39 | İLKOKUL ALANI | 161 | %0.51
42 | LİSE ALANI | 159 | %0.50
14 | KÜÇÜK SANAYİ SİTESİ | 151 | %0.47
7 | AKARYAKIT VE SERVİS İSTASYONU ALANI | 141 | %0.44
40 | ORTAOKUL ALANI | 138 | %0.43
12 | DEPOLAMA VE LOJİSTİK TESİS ALANI | 128 | %0.40
59 | BİRİNCİ BASAMAK SAĞLIK TESİSİ ALANI | 119 | %0.37
38 | ANAOKULU ALANI | 118 | %0.37
56 | ÖĞRENCİ YURDU ALANI | 113 | %0.36
27 | KAM

In [5]:
# RAPOR 1
fc = out_fc_name  # 2. hücrede oluşturulan referans dataset

ilce_profiles = {}

with arcpy.da.SearchCursor(fc, ["ilce", "SHAPE_Area", "ana_kullanim", "alt_kullanim"]) as cursor:
    for ilce, area, ana, alt in cursor:
        
        if ilce not in ilce_profiles:
            ilce_profiles[ilce] = {
                "feature_count": 0,
                "total_area": 0,
                "small_area_count": 0,
                "alt_dolu": 0,
                "ana_set": set(),
                "ana_area": defaultdict(float)
            }
        
        p = ilce_profiles[ilce]
        p["feature_count"] += 1
        p["total_area"] += area
        
        if area < 3000:
            p["small_area_count"] += 1
        
        if alt not in [None, ""]:
            p["alt_dolu"] += 1
        
        if ana:
            p["ana_set"].add(ana)
            p["ana_area"][ana] += area

print("\nILCE PROFILI\n")

for ilce, p in ilce_profiles.items():
    
    feature_count = p["feature_count"]
    total_area = p["total_area"]
    avg_area = total_area / feature_count if feature_count else 0
    small_ratio = (p["small_area_count"] / feature_count) * 100 if feature_count else 0
    alt_doluluk = (p["alt_dolu"] / feature_count) * 100 if feature_count else 0
    fonksiyon_cesit = len(p["ana_set"])
    
    sorted_funcs = sorted(p["ana_area"].items(), key=lambda x: x[1], reverse=True)
    top3_area = sum([f[1] for f in sorted_funcs[:3]])
    top3_ratio = (top3_area / total_area) * 100 if total_area else 0
    
    referans_mi = "Referans İlçe" if ilce in referans_ilceler else "Devam Ediyor"
    
    print(f"İlçe: {ilce} | Referans: {referans_mi}")
    print(f"  Parça Sayısı: {feature_count}")
    print(f"  Toplam Alan (m²): {round(total_area,2)}")
    print(f"  Ortalama Alan (m²): {round(avg_area,2)}")
    print(f"  <3000 m² Oranı (%): {round(small_ratio,2)}")
    print(f"  Alt Kullanım Doluluk (%): {round(alt_doluluk,2)}")
    print(f"  Fonksiyon Çeşitliliği: {fonksiyon_cesit}")
    print(f"  İlk 3 Fonksiyon Alan Oranı (%): {round(top3_ratio,2)}")
    print("-" * 60)



ILCE PROFILI

İlçe: 1739 | Referans: Devam Ediyor
  Parça Sayısı: 3483
  Toplam Alan (m²): 17064515.28
  Ortalama Alan (m²): 4899.37
  <3000 m² Oranı (%): 69.05
  Alt Kullanım Doluluk (%): 99.8
  Fonksiyon Çeşitliliği: 0
  İlk 3 Fonksiyon Alan Oranı (%): 0.0
------------------------------------------------------------
İlçe: 1886 | Referans: Referans İlçe
  Parça Sayısı: 3179
  Toplam Alan (m²): 13542003.14
  Ortalama Alan (m²): 4259.83
  <3000 m² Oranı (%): 61.91
  Alt Kullanım Doluluk (%): 100.0
  Fonksiyon Çeşitliliği: 0
  İlk 3 Fonksiyon Alan Oranı (%): 0.0
------------------------------------------------------------
İlçe: 2055 | Referans: Devam Ediyor
  Parça Sayısı: 2778
  Toplam Alan (m²): 55769449.65
  Ortalama Alan (m²): 20075.4
  <3000 m² Oranı (%): 22.68
  Alt Kullanım Doluluk (%): 0.18
  Fonksiyon Çeşitliliği: 0
  İlk 3 Fonksiyon Alan Oranı (%): 0.0
------------------------------------------------------------
İlçe: 1336 | Referans: Devam Ediyor
  Parça Sayısı: 3051
  Toplam

In [6]:
# RAPOR 2
ilce_kod_field = "ILCE"
area_field = "SHAPE_AREA"
alt_kullanim_field = "ALT_KULLANIM"

# REST servis URL'si
ilce_tablo_url = r"https://geo.ipa.istanbul/server/rest/services/Hosted/ilce_turef30/FeatureServer/0"   # kendi URL'ni koy

ilce_ad_kod_field = "kimlikno"
ilce_ad_field = "ad"

referans_set = set(referans_ilceler)

# --------------------------------------------------
# 1️⃣ REST tablosunu geçici layer yap
# --------------------------------------------------

ilce_layer = "ilce_layer_temp"
arcpy.management.MakeFeatureLayer(ilce_tablo_url, ilce_layer)

# --------------------------------------------------
# 2️⃣ Kod -> Ad sözlüğü oluştur
# --------------------------------------------------

ilce_ad_dict = {}

with arcpy.da.SearchCursor(ilce_layer, [ilce_ad_kod_field, ilce_ad_field]) as cursor:
    for kod, ad in cursor:
        ilce_ad_dict[kod] = ad

# --------------------------------------------------
# 3️⃣ İlçe bazlı agregasyon
# --------------------------------------------------

veri = defaultdict(lambda: {"alan":0, "dolu_alan":0})

with arcpy.da.SearchCursor(fc, [ilce_kod_field, area_field, alt_kullanim_field]) as cursor:
    for ilce, alan, alt in cursor:

        if ilce is None:
            continue

        veri[ilce]["alan"] += alan if alan else 0

        if alt not in [None, "", "YOK"]:
            veri[ilce]["dolu_alan"] += alan if alan else 0


# --------------------------------------------------
# 4️⃣ Özet ve Rapor
# --------------------------------------------------

print("\n================ İLÇE GENEL ÖZETİ ================\n")

toplam_ilce = 0
referans_sayisi = 0
devam_sayisi = 0
baslanmamis_sayisi = 0

sonuc_listesi = []

for ilce in sorted(veri.keys()):

    toplam_ilce += 1

    toplam_alan = veri[ilce]["alan"]
    dolu_alan = veri[ilce]["dolu_alan"]

    doluluk_orani = (dolu_alan / toplam_alan * 100) if toplam_alan > 0 else 0

    if ilce in referans_set:
        durum = "Referans İlçe"
        referans_sayisi += 1
    elif doluluk_orani < 5:
        durum = "Başlanmadı"
        baslanmamis_sayisi += 1
    else:
        durum = "Devam Ediyor"
        devam_sayisi += 1

    ilce_adi = ilce_ad_dict.get(ilce, "Bilinmiyor")

    sonuc_listesi.append((ilce, ilce_adi, durum, doluluk_orani))


print(f"Toplam İlçe Sayısı     : {toplam_ilce}")
print(f"Referans İlçe Sayısı   : {referans_sayisi}")
print(f"Devam Eden İlçe Sayısı : {devam_sayisi}")
print(f"Başlanmamış İlçe Sayısı: {baslanmamis_sayisi}")

print("\n================ İLÇE PROFİLİ ================\n")

for kod, ad, durum, oran in sonuc_listesi:

    print(f"{kod} - {ad}")
    print(f"Durum: {durum}")
    print(f"Doluluk Oranı (%): {round(oran,2)}")
    print("-" * 40)



================ İLÇE GENEL ÖZETİ ================

Toplam İlçe Sayısı     : 39
Referans İlçe Sayısı   : 6
Devam Eden İlçe Sayısı : 26
Başlanmamış İlçe Sayısı: 7

================ İLÇE PROFİLİ ================

1103 - ADALAR
Durum: Başlanmadı
Doluluk Oranı (%): 2.27
----------------------------------------
1166 - BAKIRKÖY
Durum: Devam Ediyor
Doluluk Oranı (%): 100.0
----------------------------------------
1183 - BEŞİKTAŞ
Durum: Devam Ediyor
Doluluk Oranı (%): 100.0
----------------------------------------
1185 - BEYKOZ
Durum: Devam Ediyor
Doluluk Oranı (%): 18.47
----------------------------------------
1186 - BEYOĞLU
Durum: Devam Ediyor
Doluluk Oranı (%): 12.92
----------------------------------------
1237 - ÇATALCA
Durum: Devam Ediyor
Doluluk Oranı (%): 36.45
----------------------------------------
1325 - EYÜPSULTAN
Durum: Devam Ediyor
Doluluk Oranı (%): 14.68
----------------------------------------
1327 - FATİH
Durum: Başlanmadı
Doluluk Oranı (%): 4.24
--------------------------

In [8]:
df_kentsel = df[df["ana_kullanim"] == "KENTSEL"]

<class 'NameError'>: name 'df' is not defined

In [9]:
# -------------------------------------------------
# 1️⃣ Tarımsal mahalle Excelini oku
# -------------------------------------------------

excel_path = r"C:\Users\harun.korucu\Documents\ArcGIS\Projects\MyProject\222_tarimsal_mahalle.xlsx"

df_tarimsal = pd.read_excel(excel_path)

print("Tarımsal Excel kolonları:")
print(df_tarimsal.columns.tolist())


# -------------------------------------------------
# 2️⃣ Merge için kolonları standardize et
# -------------------------------------------------

# Mahalle isimlerini büyük harfe çevirip boşlukları temizleyelim
df_kentsel["mahalle_std"] = df_kentsel["mahalle"].str.strip().str.upper()
df_tarimsal["mahalle_std"] = df_tarimsal["KOY_ADI"].str.strip().str.upper()   # ← Exceldeki mahalle kolon adı farklıysa


# -------------------------------------------------
# 3️⃣ Merge (Left Join)
# -------------------------------------------------

df_merged = df_kentsel.merge(
    df_tarimsal[["mahalle_std"]],   # sadece eşleşme kolonu yeterli
    on="mahalle_std",
    how="left",
    indicator=True
)

print(df_merged["_merge"].value_counts())


# -------------------------------------------------
# 4️⃣ Tarımsal olanları exclude et
# -------------------------------------------------

df_kentsel_temiz = df_merged[df_merged["_merge"] == "left_only"].copy()

print("Exclude sonrası kayıt sayısı:", len(df_kentsel_temiz))


Tarımsal Excel kolonları:
['ILCE_ADI', 'KOY_ADI', 'UAVT', 'Kirsal_Mahalle_Koru']


<class 'NameError'>: name 'df_kentsel' is not defined

In [8]:
# -------------------------------------------------
# STATUS üret
# -------------------------------------------------

df_kentsel_temiz["STATUS"] = df_kentsel_temiz["alt_kullanim"].apply(
    lambda x: "Tamamlandı" if pd.notnull(x) and str(x).strip() != "" else "Devam Ediyor"
)


<class 'NameError'>: name 'df_kentsel_temiz' is not defined

In [51]:
%whos

Variable                Type             Data/Info
--------------------------------------------------
Counter                 type             <class 'collections.Counter'>
FeatureLayer            type             <class 'arcgis.features.layer.FeatureLayer'>
GIS                     type             <class 'arcgis.gis.GIS'>
GeoAccessor             type             <class 'arcgis.features.g<...>o._accessor.GeoAccessor'>
aciklama                str              LİMAN
ad                      str              SULTANGAZİ
alan                    float            0.0
alt                     NoneType         None
alt_doluluk             float            1.69971671388102
alt_domain              dict             n=131
alt_kullanim            str              126
alt_kullanim_field      str              ALT_KULLANIM
ana                     NoneType         None
arcpy                   module           <module 'arcpy' from 'C:\<...>cPy\\arcpy\\__init__.py'>
area                    float            

In [49]:
# -------------------------------------------------
# İlçe bazında tamamlanma özeti
# -------------------------------------------------

ozet_ilce = (
    df_kentsel_temiz
    .groupby("ilce")
    .agg(
        Toplam=("STATUS", "count"),
        Tamamlanan=("STATUS", lambda x: (x == "Tamamlandı").sum())
    )
    .reset_index()
)

# Yüzde hesap
ozet_ilce["Tamamlanma_Yuzde"] = (
    ozet_ilce["Tamamlanan"] / ozet_ilce["Toplam"] * 100
).round(2)

# Büyükten küçüğe sırala
ozet_ilce = ozet_ilce.sort_values("Tamamlanma_Yuzde", ascending=False)

ozet_ilce


,ilce,Toplam,Tamamlanan,Tamamlanma_Yuzde
36,2053,7609,7609,100.00
26,2010,2569,2569,100.00
25,2005,3764,3764,100.00
24,2004,6497,6497,100.00
22,1886,3179,3179,100.00
15,1708,5477,5477,100.00
1,1166,3077,3076,99.97
2,1183,2852,2845,99.75
20,1835,8765,8742,99.74
16,1739,3469,3457,99.65


In [55]:
# -------------------------------------------------
# 1️⃣ Portal bağlantısı
# -------------------------------------------------

from arcgis.gis import GIS
from arcgis.features import FeatureLayer

gis = GIS("home")

# ilce_turef_30 layer URL'sini yaz
ilce_layer = FeatureLayer("https://geo.ipa.istanbul/server/rest/services/Hosted/ilce_turef30/FeatureServer/0")

# Sadece gerekli alanları çek
df_ilce = ilce_layer.query(
    where="1=1",
    out_fields="kimlikno, ad",
    as_df=True
)

print("Portal ilçe kayıt sayısı:", len(df_ilce))


# -------------------------------------------------
# 2️⃣ Veri tiplerini eşitle
# -------------------------------------------------

ozet_ilce["ilce"] = ozet_ilce["ilce"].astype(str)
df_ilce["kimlikno"] = df_ilce["kimlikno"].astype(str)


# -------------------------------------------------
# 3️⃣ Merge işlemi
# -------------------------------------------------

ozet_ilce = ozet_ilce.merge(
    df_ilce,
    left_on="ilce",
    right_on="kimlikno",
    how="left"
)


# -------------------------------------------------
# 4️⃣ İlçe adını güncelle
# -------------------------------------------------

ozet_ilce["ilce"] = ozet_ilce["ad"]

ozet_ilce = ozet_ilce.drop(columns=["kimlikno", "ad"])


# -------------------------------------------------
# 5️⃣ Son hali
# -------------------------------------------------

ozet_ilce = ozet_ilce.sort_values("Tamamlanma_Yuzde", ascending=False)

ozet_ilce


Portal ilçe kayıt sayısı: 39


,ilce,Toplam,Tamamlanan,Tamamlanma_Yuzde,objectid,SHAPE
36,ESENYURT,7609,7609,100.00,39,"{""rings"": [[[390637.4866000004, 4542087.233100..."
26,GÜNGÖREN,2569,2569,100.00,4,"{""rings"": [[[405202.14070000034, 4545460.15709..."
25,BAHÇELİEVLER,3764,3764,100.00,10,"{""rings"": [[[401053.2879999997, 4543443.4958],..."
24,BAĞCILAR,6497,6497,100.00,13,"{""rings"": [[[401430.84810000006, 4543700.31729..."
22,BAYRAMPAŞA,3179,3179,100.00,35,"{""rings"": [[[406103.9128999999, 4549515.882999..."
15,ÜSKÜDAR,5477,5477,100.00,37,"{""rings"": [[[421484.6947999997, 4549571.268300..."
1,BAKIRKÖY,3077,3076,99.97,27,"{""rings"": [[[397104.35979999974, 4539737.0836]..."
2,BEŞİKTAŞ,2852,2845,99.75,38,"{""rings"": [[[418081.3668999998, 4552006.1658],..."
20,PENDİK,8765,8742,99.74,20,"{""rings"": [[[446552.6478000004, 4546957.92], [..."
16,ZEYTİNBURNU,3469,3457,99.65,7,"{""rings"": [[[408626.57270000037, 4544657.58919..."
